# Rust 119: PostgreSQL — Practice

Companion exercises for `19_postgresql_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and every demonstration cell has an exercise, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the `todo!()` bodies; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. Connecting: Options and the Pool](#1-connecting-options-and-the-pool)
- [2. Queries: fetch_all, fetch_optional, fetch_one](#2-queries-fetch_all-fetch_optional-fetch_one)
- [3. Rows to Structs: FromRow](#3-rows-to-structs-fromrow)
- [4. NULL and Option](#4-null-and-option)
- [5. Transactions Across Await Points](#5-transactions-across-await-points)
- [6. Bind Parameters and SQL Injection](#6-bind-parameters-and-sql-injection)
- [7. The Pool: Sizing, Acquire Timeouts](#7-the-pool-sizing-acquire-timeouts)
- [8. Migrations: Schema as Code](#8-migrations-schema-as-code)
- [9. sqlx in the Ecosystem](#9-sqlx-in-the-ecosystem)
- [10. Mixed Review](#10-mixed-review)
- [11. What Comes Next](#11-what-comes-next)

§1–§9 each have exercises for the demonstration cells of the example notebook; §10 mixes corners. §10 of the example notebook is the E-code catalogue.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from an earlier cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A struct or fn may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers. The same rule covers `use` lines: the first cell that needs a name is the one that imports; later cells rely on persistence — a repeated `use` is E0252. And a trailing `//` comment after a `use` line's semicolon hides that semicolon from the tooling — put the comment on its own line above.

### The stub convention

Each exercise cell declares signatures with contract doc comments and a `// --- check (do not modify) ---` block of asserts. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Four notes specific to this chapter

- **Every check builds its own desk.** The given `pg_desk()` helper (§1.1) starts a fresh wire-protocol server on an ephemeral port and returns its port; `pool_on(port)` is the one-true pool builder both notebooks use. Nothing here touches a real network.
- **Stubs take the port, build the pool.** A stub is an `async fn` receiving `port: u16`; it builds its pool with the given `pool_on`, does the work, and returns plain values the check can assert. Never hold a pool across cells — the desk counts on connections closing. The §7.1 and §10.4 exceptions build their pools with `PgPoolOptions` directly — the 1-connection cap and 200ms timeout are the exercise.
- **Name decode types at the call site.** `row.get::<i32, _>("id")`, `let v: Option<String> = ...` — un-annotated decodes are E0282/E0283, the same shape §10.1 of the example notebook taught.
- **The crate cell below is given and must run first** (once per kernel session, features included). A `cannot find crate` diagnostic means it has not run — restart and go from the top.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines serving a stub you have not solved yet;
- **stub-induced dead code** (`never used` / `never constructed`) — given helpers that only a *solution* calls.

Chapter 14's fourth family (`variable does not need to be mutable`) is designed out: no stub takes a `mut` binding parameter — connections come from `pool_on(port)` inside the stub.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session, features included.
:dep sqlx = { version = "0.8", default-features = false, features = ["runtime-tokio", "postgres"] }
:dep tokio = { version = "1", features = ["full"] }

let rt = tokio::runtime::Runtime::new().unwrap();
let answer = rt.block_on(async { 40 + 2 });
assert_eq!(answer, 42);
println!("crates loaded; block_on smoke: {}", answer);


## 1. Connecting: Options and the Pool

`PgConnectOptions` configures; `PgPoolOptions` sizes; `connect_with` returns a pool that clones freely and opens connections lazily.

**See**: §1 of the example notebook.


In [ ]:
// 1.1 Given: the local PostgreSQL desk — the example notebook's §0.2 verbatim
// (40 cells of wire protocol, compressed here to the same two entry points).
// Every exercise calls pg_desk() for a port of its own, then pool_on(port).
//
// The full server source lives in the example notebook's §0.2; this helper
// embeds the same binary behavior via a tiny local re-implementation so the
// practice notebook stands alone. Do not modify.

use std::collections::HashMap;
use std::io::{Read, Write};
use std::sync::{Arc, Mutex};

fn msg(tag: u8, body: &[u8]) -> Vec<u8> {
    let mut out = vec![tag];
    out.extend_from_slice(&((body.len() + 4) as i32).to_be_bytes());
    out.extend_from_slice(body);
    out
}
fn cstr(s: &str) -> Vec<u8> { let mut v = s.as_bytes().to_vec(); v.push(0); v }
fn i32be(n: i32) -> Vec<u8> { n.to_be_bytes().to_vec() }
fn i16be(n: i16) -> Vec<u8> { n.to_be_bytes().to_vec() }
fn authentication_ok() -> Vec<u8> { msg(b'R', &i32be(0)) }
fn parameter_status(k: &str, v: &str) -> Vec<u8> { let mut b = cstr(k); b.extend(cstr(v)); msg(b'S', &b) }
fn ready_for_query(status: u8) -> Vec<u8> { msg(b'Z', &[status, 0]) }
fn command_complete(tag: &str) -> Vec<u8> { msg(b'C', &cstr(tag)) }
fn error_response(code: &str, message: &str) -> Vec<u8> {
    let mut b = vec![b'S']; b.extend(cstr("ERROR")); b.push(b'V'); b.extend(cstr("ERROR"));
    b.push(b'C'); b.extend(cstr(code)); b.push(b'M'); b.extend(cstr(message)); b.push(0);
    msg(b'E', &b)
}
fn parse_complete() -> Vec<u8> { msg(b'1', &[]) }
fn bind_complete() -> Vec<u8> { msg(b'2', &[]) }
fn no_data() -> Vec<u8> { msg(b'n', &[]) }
fn empty_query_response() -> Vec<u8> { msg(b'I', &[]) }

const INT4: i32 = 23;
const TEXT: i32 = 25;
struct Col { name: &'static str, oid: i32, fmt: i16 }
fn int4col(name: &'static str) -> Col { Col { name, oid: INT4, fmt: 1 } }
fn textcol(name: &'static str) -> Col { Col { name, oid: TEXT, fmt: 0 } }
fn row_description(cols: &[Col]) -> Vec<u8> {
    let mut b = i16be(cols.len() as i16);
    for c in cols {
        b.extend(cstr(c.name)); b.extend(i32be(0)); b.extend(i16be(0)); b.extend(i32be(c.oid));
        b.extend(i16be(if c.oid == INT4 { 4 } else { -1 })); b.extend(i32be(-1)); b.extend(i16be(c.fmt));
    }
    msg(b'T', &b)
}
type Cell = Option<Vec<u8>>;
type WireRow = Vec<Cell>;
fn data_row(row: &[Cell]) -> Vec<u8> {
    let mut b = i16be(row.len() as i16);
    for cell in row {
        match cell {
            Some(v) => { b.extend(i32be(v.len() as i32)); b.extend(v); }
            None => b.extend(i32be(-1)),
        }
    }
    msg(b'D', &b)
}

#[derive(Default)]
struct Db {
    rows: Vec<(i32, Option<String>)>,
    next_id: i32,
    snapshot: Option<(Vec<(i32, Option<String>)>, i32)>,
}
impl Db {
    fn tx_note(&self, kind: &str) {
        use std::io::Write;
        if let Ok(mut f) = std::fs::OpenOptions::new().create(true).append(true).open("pg_tx_p.log") {
            writeln!(f, "{}", kind).ok();
        }
    }
}
type Shared = Arc<Mutex<Db>>;

fn run_query(db: &mut Db, sql: &str, params: &[Option<String>], in_tx: bool) -> (Vec<u8>, bool) {
    let q = sql.trim();
    let ql = q.to_lowercase();
    if q.is_empty() { return (empty_query_response(), in_tx); }
    let err = |code: &str, m: &str| (error_response(code, m), in_tx);
    let p0 = || params.first().and_then(|p| p.clone());
    if ql == "begin" || ql.starts_with("begin ") {
        if in_tx { return err("25001", "there is already a transaction in progress"); }
        db.snapshot = Some((db.rows.clone(), db.next_id));
        db.tx_note("BEGIN");
        return (command_complete("BEGIN"), true);
    }
    if ql == "commit" || ql == "commit work" || ql == "end" {
        if !in_tx { return err("25P01", "there is no transaction in progress"); }
        db.snapshot = None;
        db.tx_note("COMMIT");
        return (command_complete("COMMIT"), false);
    }
    if ql.starts_with("rollback") {
        if !in_tx { return err("25P01", "there is no transaction in progress"); }
        if let Some((rows, next)) = db.snapshot.take() { db.rows = rows; db.next_id = next; }
        db.tx_note("ROLLBACK");
        return (command_complete("ROLLBACK"), false);
    }
    if ql == "select 1" || ql == "select 1;" {
        let cols = [int4col("n")];
        let mut out = row_description(&cols);
        out.extend(data_row(&[Some(i32be(1))]));
        out.extend(command_complete("SELECT 1"));
        return (out, in_tx);
    }
    if ql.starts_with("select count(*)") {
        let cols = [int4col("count")];
        let mut out = row_description(&cols);
        out.extend(data_row(&[Some(i32be(db.rows.len() as i32))]));
        out.extend(command_complete("SELECT 1"));
        return (out, in_tx);
    }
    if ql.starts_with("select") {
        if !ql.contains("from t") && !ql.contains("from fills") {
            let table = ql.split_whitespace().skip_while(|w| *w != "from").nth(1).unwrap_or("?").trim_end_matches(';');
            return err("42P01", &format!("relation \"{}\" does not exist", table));
        }
        let cols = [int4col("id"), textcol("name")];
        let filter = if ql.contains("where name = $1") { p0() } else { None };
        let mut rs: Vec<WireRow> = db.rows.iter()
            .filter(|(_, name)| match &filter {
                Some(f) => name.as_deref() == Some(f.as_str()),
                None => true,
            })
            .map(|(id, name)| vec![Some(i32be(*id)), name.as_ref().map(|n| n.as_bytes().to_vec())])
            .collect();
        rs.sort_by(|a, b| match (&a[0], &b[0]) {
            (Some(x), Some(y)) => x.cmp(y),
            _ => std::cmp::Ordering::Equal,
        });
        let tag = format!("SELECT {}", rs.len());
        let mut out = row_description(&cols);
        for r in &rs { out.extend(data_row(r)); }
        out.extend(command_complete(&tag));
        return (out, in_tx);
    }
    if ql.starts_with("insert") {
        let id = db.next_id;
        db.next_id += 1;
        db.rows.push((id, p0()));
        return (command_complete("INSERT 0 1"), in_tx);
    }
    if ql.starts_with("update") {
        let nv = p0();
        let mut n = 0;
        for r in db.rows.iter_mut() { r.1 = nv.clone(); n += 1; }
        return (command_complete(&format!("UPDATE {}", n)), in_tx);
    }
    if ql.starts_with("delete") {
        let n = db.rows.len();
        db.rows.clear();
        return (command_complete(&format!("DELETE {}", n)), in_tx);
    }
    err("42601", &format!("syntax error at or near \"{}\"", &q[..q.len().min(24)]))
}

fn returns_rows(sql: &str) -> bool { sql.trim().to_lowercase().starts_with("select") }
fn cstring_at(body: &[u8], from: usize) -> String {
    let mut end = from;
    while end < body.len() && body[end] != 0 { end += 1; }
    String::from_utf8_lossy(&body[from..end]).into_owned()
}
struct Conn {
    statements: HashMap<String, String>,
    portals: HashMap<String, (String, Vec<Option<String>>)>,
    in_tx: bool,
}
fn handle(mut stream: std::net::TcpStream, db: Shared) {
    let _ = stream.set_nodelay(true);
    let mut buf: Vec<u8> = Vec::new();
    let mut tmp = [0u8; 16384];
    loop {
        if buf.len() >= 8 {
            let l = i32::from_be_bytes([buf[0], buf[1], buf[2], buf[3]]) as usize;
            if buf.len() >= l { break; }
        }
        match stream.read(&mut tmp) {
            Ok(0) | Err(_) => return,
            Ok(n) => buf.extend_from_slice(&tmp[..n]),
        }
    }
    let code = i32::from_be_bytes([buf[4], buf[5], buf[6], buf[7]]);
    if code == 80877103 {
        let _ = stream.write_all(b"N");
        buf.drain(..8);
        loop {
            if buf.len() >= 8 {
                let l = i32::from_be_bytes([buf[0], buf[1], buf[2], buf[3]]) as usize;
                if buf.len() >= l { break; }
            }
            match stream.read(&mut tmp) {
                Ok(0) | Err(_) => return,
                Ok(n) => buf.extend_from_slice(&tmp[..n]),
            }
        }
    }
    let len = i32::from_be_bytes([buf[0], buf[1], buf[2], buf[3]]) as usize;
    let protocol = i32::from_be_bytes([buf[4], buf[5], buf[6], buf[7]]);
    buf.drain(..len);
    if protocol != 196608 {
        stream.write_all(&error_response("28000", "unsupported protocol")).ok();
        return;
    }
    let mut out = Vec::new();
    out.extend(authentication_ok());
    for (k, v) in [("server_version", "16.9"), ("client_encoding", "UTF8"), ("DateStyle", "ISO, MDY"), ("integer_datetimes", "on"), ("TimeZone", "UTC"), ("standard_conforming_strings", "on")] {
        out.extend(parameter_status(k, v));
    }
    out.extend(ready_for_query(b'I'));
    if stream.write_all(&out).is_err() { return; }
    stream.flush().ok();
    let mut conn = Conn { statements: HashMap::new(), portals: HashMap::new(), in_tx: false };
    loop {
        while buf.len() < 5 {
            match stream.read(&mut tmp) {
                Ok(0) | Err(_) => return,
                Ok(n) => buf.extend_from_slice(&tmp[..n]),
            }
        }
        let want = 1 + i32::from_be_bytes([buf[1], buf[2], buf[3], buf[4]]) as usize;
        while buf.len() < want {
            match stream.read(&mut tmp) {
                Ok(0) | Err(_) => return,
                Ok(n) => buf.extend_from_slice(&tmp[..n]),
            }
        }
        let tag = buf[0];
        let mlen = i32::from_be_bytes([buf[1], buf[2], buf[3], buf[4]]) as usize;
        let total = 1 + mlen;
        if mlen < 4 { continue; }
        let body = if mlen > 4 { buf[5..total].to_vec() } else { Vec::new() };
        buf.drain(..total);
        let mut reply: Vec<u8> = Vec::new();
        match tag {
            b'Q' => {
                let query = cstring_at(&body, 0);
                let mut d = db.lock().unwrap();
                let (payload, tx) = run_query(&mut d, &query, &[], conn.in_tx);
                conn.in_tx = tx;
                reply.extend(payload);
                reply.extend(ready_for_query(if tx { b'T' } else { b'I' }));
            }
            b'P' => {
                let name = cstring_at(&body, 0);
                let sql = cstring_at(&body, name.len() + 1);
                conn.statements.insert(name, sql);
                reply.extend(parse_complete());
            }
            b'B' => {
                let portal = cstring_at(&body, 0);
                let sstart = portal.len() + 1;
                let stmt = cstring_at(&body, sstart);
                let sql = conn.statements.get(&stmt).cloned().unwrap_or_default();
                let mut p = sstart + stmt.len() + 1;
                let mut params: Vec<Option<String>> = Vec::new();
                if p + 2 <= body.len() {
                    let nfmt = i16::from_be_bytes([body[p], body[p + 1]]) as usize;
                    p += 2 + nfmt * 2;
                    if p + 2 <= body.len() {
                        let nvals = i16::from_be_bytes([body[p], body[p + 1]]) as usize;
                        p += 2;
                        for _ in 0..nvals {
                            if p + 4 > body.len() { break; }
                            let vlen = i32::from_be_bytes([body[p], body[p + 1], body[p + 2], body[p + 3]]);
                            p += 4;
                            if vlen < 0 { params.push(None); }
                            else {
                                let end = (p + vlen as usize).min(body.len());
                                params.push(Some(String::from_utf8_lossy(&body[p..end]).into_owned()));
                                p = end;
                            }
                        }
                    }
                }
                conn.portals.insert(portal, (sql, params));
                reply.extend(bind_complete());
            }
            b'D' => {
                let kind = body[0];
                let name = cstring_at(&body, 1);
                let sql = match kind {
                    b'S' => conn.statements.get(&name).cloned().unwrap_or_default(),
                    _ => conn.portals.get(&name).map(|(s, _)| s.clone()).unwrap_or_default(),
                };
                reply.extend(msg(b't', &i16be(0)));
                if returns_rows(&sql) {
                    if sql.trim().to_lowercase() == "select 1" {
                        reply.extend(row_description(&[int4col("n")]));
                    } else if sql.to_lowercase().contains("count(*)") {
                        reply.extend(row_description(&[int4col("count")]));
                    } else {
                        reply.extend(row_description(&[int4col("id"), textcol("name")]));
                    }
                } else {
                    reply.extend(no_data());
                }
            }
            b'E' => {
                let portal = cstring_at(&body, 0);
                let (sql, params) = conn.portals.get(&portal).cloned().unwrap_or_default();
                let mut d = db.lock().unwrap();
                let (payload, tx) = run_query(&mut d, &sql, &params, conn.in_tx);
                conn.in_tx = tx;
                reply.extend(payload);
            }
            b'S' => { reply.extend(ready_for_query(if conn.in_tx { b'T' } else { b'I' })); }
            b'C' => { reply.extend(msg(b'3', &[])); }
            b'H' => {}
            b'X' => {
                if conn.in_tx {
                    let mut d = db.lock().unwrap();
                    if let Some((rows, next)) = d.snapshot.take() { d.rows = rows; d.next_id = next; }
                    d.tx_note("ROLLBACK(dropped)");
                }
                return;
            }
            _ => {
                reply.extend(error_response("08P01", "protocol violation"));
                reply.extend(ready_for_query(if conn.in_tx { b'T' } else { b'I' }));
            }
        }
        if reply.is_empty() { continue; }
        if stream.write_all(&reply).is_err() { return; }
        stream.flush().ok();
    }
}

/// Start a fresh desk; return its port. Given.
fn pg_desk() -> u16 {
    let listener = std::net::TcpListener::bind("127.0.0.1:0").unwrap();
    let port = listener.local_addr().unwrap().port();
    let db: Shared = Arc::new(Mutex::new(Db { next_id: 1, ..Db::default() }));
    std::thread::spawn(move || {
        for stream in listener.incoming().flatten() {
            let db = Arc::clone(&db);
            std::thread::spawn(move || handle(stream, db));
        }
    });
    port
}

/// The one-true pool builder: every exercise starts here. Given.
async fn pool_on(port: u16) -> sqlx::PgPool {
    let opts = sqlx::postgres::PgConnectOptions::new()
        .host("127.0.0.1")
        .port(port)
        .username("desk")
        .database("desk");
    sqlx::postgres::PgPoolOptions::new()
        .max_connections(5)
        .acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts)
        .await
        .unwrap()
}

/// Connect to `port` and report the pool's size after connect. The whole
/// builder chain is the exercise.
async fn open_pool(port: u16) -> u32 {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
let size = rt.block_on(open_pool(port));
assert!(size >= 1, "the pool opened at least one connection");
println!("Exercise 1.1 passed");


## 2. Queries: fetch_all, fetch_optional, fetch_one

Three shapes, one API. **`fetch_all`** returns a `Vec` — zero rows is an empty vec, never an error; **`fetch_optional`** returns `Option` — `None` for an *empty result*, while a SQL error is still `Err`; **`fetch_one`** demands exactly one row and errors with `RowNotFound` on empty. Writes go through `execute` and report **`rows_affected()`**.

**See**: §2 of the example notebook.


In [ ]:
// 2.1 The three fetch shapes, the write receipt, and the empty-vs-error line.
// Row's get accessors: imported once, here, persisted below
use sqlx::Row;

/// INSERT `name` into the desk table as a bound parameter; return
/// rows_affected — the write receipt.
async fn insert_named(port: u16, name: &str) -> u64 {
    todo!()
}

/// fetch_all every row of the table and return the ids (the desk returns
/// them id-ordered). An empty table would be an empty vec, never an error.
async fn all_ids(port: u16) -> Vec<i32> {
    todo!()
}

/// fetch_optional the first row of the whole table; None only when the
/// table is empty.
async fn first_id(port: u16) -> Option<i32> {
    todo!()
}

/// fetch_optional the row whose name is exactly `name`: None is an EMPTY
/// RESULT, never an error.
async fn lookup(port: u16, name: &str) -> Option<i32> {
    todo!()
}

/// Run `SELECT * FROM missing` and return the server's error code — the
/// machine-readable half of a Database error (an Err, not a None).
async fn error_code(port: u16) -> String {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    assert_eq!(insert_named(port, "ana").await, 1);
    assert_eq!(insert_named(port, "bob").await, 1);
    assert_eq!(all_ids(port).await, vec![1, 2], "fetch_all: zero rows would be a vec, never an error");
    assert_eq!(first_id(port).await, Some(1));
    assert_eq!(lookup(port, "bob").await, Some(2), "an empty result is None");
    assert_eq!(lookup(port, "ghost").await, None);
    assert_eq!(error_code(port).await, "42P01", "a SQL error is Err, not None — undefined table");
});
println!("Exercise 2.1 passed");


## 3. Rows to Structs: FromRow

`PgRow` is the raw contract; your struct is the typed one; **`FromRow` is the bridge**. With the `macros` feature off you write the impl by hand — one `try_get` per field — and **`query_as`** decodes every row through it. The destination type lives in the binding, and tuples are destinations too.

**See**: §3 of the example notebook.


In [ ]:
// 3.1 query_as decodes through a hand-written FromRow.
// Given: the Fill row type and its impl, verbatim from §3 of the example
// notebook. Do not modify.
#[derive(Debug, PartialEq)]
struct Fill {
    id: i32,
    name: Option<String>,
}

impl<'r> sqlx::FromRow<'r, sqlx::postgres::PgRow> for Fill {
    fn from_row(row: &'r sqlx::postgres::PgRow) -> Result<Self, sqlx::Error> {
        Ok(Fill {
            id: row.try_get("id")?,
            name: row.try_get("name")?,
        })
    }
}

/// Decode every row of `SELECT * FROM t` into `Fill`s — `query_as` is
/// generic over the destination, which this function's return type names.
async fn decode_fills(port: u16) -> Vec<Fill> {
    todo!()
}

/// The same query with NO struct: tuples are FromRow too. Decode into
/// (id, name) pairs.
async fn decode_pairs(port: u16) -> Vec<(i32, Option<String>)> {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    insert_named(port, "ana").await;
    insert_named(port, "bob").await;
    let fills = decode_fills(port).await;
    assert_eq!(fills.len(), 2, "query_as decoded through the hand-written impl");
    assert_eq!(fills[0].id, 1);
    assert_eq!(fills[0].name.as_deref(), Some("ana"));
    let pairs = decode_pairs(port).await;
    assert_eq!(pairs[1], (2, Some("bob".to_string())), "tuples decode too — same generic destination move");
});
println!("Exercise 3.1 passed");


## 4. NULL and Option

The schema's nullability is part of its type, and **`Option` is the only honest translation**. Bind `Option::<String>::None` to store a NULL; decode `Option<String>` to read one; a bare `String` refuses at run time with `ColumnDecode` — the schema said *maybe absent*, so the type must say it too.

**See**: §4 of the example notebook.


In [ ]:
// 4.1 Insert a NULL, read it back as None; the wrong type refuses.

/// INSERT a row whose name is NULL: bind the ABSENT value (not the empty
/// string) and return rows_affected.
async fn insert_null(port: u16) -> u64 {
    todo!()
}

/// fetch_all every row's name as Option<String>: None where NULL.
async fn all_names(port: u16) -> Vec<Option<String>> {
    todo!()
}

/// Attempt `SELECT id, name FROM t` decoded into bare (i32, String) tuples.
/// Return the outcome's name: "ColumnDecode" when the NULL row refuses the
/// bare String, "none" if everything decoded.
async fn bare_string_refusal(port: u16) -> String {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    assert_eq!(insert_null(port).await, 1);
    insert_named(port, "ana").await;
    let names = all_names(port).await;
    assert_eq!(names.len(), 2);
    assert!(names.contains(&None), "NULL decoded to None");
    assert!(names.contains(&Some("ana".to_string())));
    assert_eq!(bare_string_refusal(port).await, "ColumnDecode",
        "the schema said nullable, so bare String refuses");
});
println!("Exercise 4.1 passed");


## 5. Transactions Across Await Points

`pool.begin()` hands you the connection as a **`Transaction`** value, and the value IS the exit: `.commit()` makes the work permanent, `.rollback()` undoes it, and **dropping it without committing issues the rollback for you**. Queries inside borrow the connection through the transaction with `&mut *tx`.

**See**: §5 of the example notebook.


In [ ]:
// 5.1 The three exits, each with a count receipt.

/// Begin a transaction, INSERT `name` inside it (`&mut *tx`), and COMMIT.
/// Return the table's row count after the commit — the receipt that the
/// row survived.
async fn committed_insert(port: u16, name: &str) -> i32 {
    todo!()
}

/// Begin a transaction, INSERT `name` inside it, then DROP the Transaction
/// without committing. Wait 250ms for the automatic rollback to land, and
/// return the row count — the receipt that the ghost row is gone.
async fn dropped_insert(port: u16, name: &str) -> i32 {
    todo!()
}

/// Begin a transaction, INSERT `name` inside it, then ROLLBACK explicitly
/// (rollback takes the Transaction by value). Return the row count.
async fn rolled_back_insert(port: u16, name: &str) -> i32 {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    assert_eq!(committed_insert(port, "tx-kept").await, 1, "commit made it permanent");
    assert_eq!(dropped_insert(port, "tx-ghost").await, 1, "the dropped transaction rolled itself back");
    assert_eq!(rolled_back_insert(port, "tx-undone").await, 1, "explicit rollback undid the insert");
    assert_eq!(committed_insert(port, "tx-kept-2").await, 2, "the desk still counts the kept rows");
});
println!("Exercise 5.1 passed");


## 6. Bind Parameters and SQL Injection

Values travel **beside** the statement, never inside it: `$1` placeholders plus `.bind(...)` — the extended protocol's Parse/Bind split. A bound value cannot change a statement's shape, so the classic injection string is just data looking for a column.

**See**: §6 of the example notebook.


In [ ]:
// 6.1 The hostile string, bound and harmless.

/// INSERT `hostile` as a bound parameter, then SELECT the row back BY THE
/// SAME bound parameter. Return the stored name — verbatim.
async fn store_hostile(port: u16, hostile: &str) -> String {
    todo!()
}

/// The receipt that the table survived: the row count of `t`.
async fn table_alive(port: u16) -> i32 {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let hostile = "x'; DROP TABLE t; --";
    let stored = store_hostile(port, hostile).await;
    assert_eq!(stored, hostile, "the whole string is DATA, stored verbatim");
    assert_eq!(table_alive(port).await, 1, "no injection: the table is intact");
});
println!("Exercise 6.1 passed");


## 7. The Pool: Sizing, Acquire Timeouts

Two knobs are production decisions: **`max_connections`** caps the sockets, **`acquire_timeout`** bounds the wait — a caller past the cap gets `PoolTimedOut`, which is backpressure, not breakage. These exercises build their pools with `PgPoolOptions` directly, because the knobs ARE the exercise.

**See**: §7 of the example notebook.


In [ ]:
// 7.1 A 1-connection pool, one held checkout, one timed-out acquire.
// Built with PgPoolOptions directly — pool_on's defaults would hide the knobs

/// Build a 1-connection pool on `port` (acquire_timeout 200ms), `acquire()`
/// its only connection, then time a second `acquire()`. Return
/// (waited_ms, timed_out) — timed_out true when the error flavor is
/// sqlx::Error::PoolTimedOut.
async fn timed_acquire(port: u16) -> (u128, bool) {
    todo!()
}

/// The release side of the same story: hold the only connection, let it go,
/// and report whether the next acquire succeeds.
async fn recovers_after_release(port: u16) -> bool {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let (waited, timed_out) = timed_acquire(port).await;
    assert!(timed_out, "the error flavor is PoolTimedOut");
    assert!(waited >= 190, "the wait ran at least to the 200ms budget, got {}ms", waited);
    assert!(waited < 5000, "and it did fire rather than hang, got {}ms", waited);
    assert!(recovers_after_release(port).await, "releasing the checkout frees the socket");
});
println!("Exercise 7.1 passed");


## 8. Migrations: Schema as Code

Migrations are ordered SQL applied **exactly once, each in a transaction**, with a ledger recording what ran. The desk's stand-in ledger is the given `P_APPLIED` static; the schema work itself is a transactional `SELECT 1`.

**See**: §8 of the example notebook.


In [ ]:
// 8.1 The ledger: ordered, transactional, applied exactly once.
static P_APPLIED: std::sync::Mutex<Vec<&'static str>> = std::sync::Mutex::new(Vec::new());

/// For each version in `versions` NOT yet in the given `P_APPLIED` ledger:
/// run `SELECT 1` inside `pool.begin()`/`commit()` (the transactional
/// stand-in for DDL), then record the version in the ledger. Return the
/// versions THIS call applied, in order — a second call returns an empty
/// vec.
async fn apply_migrations(port: u16, versions: &[&'static str]) -> Vec<&'static str> {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
let migs: &[&'static str] = &["0001_init", "0002_indexes"];
rt.block_on(async {
    let first = apply_migrations(port, migs).await;
    assert_eq!(first, vec!["0001_init", "0002_indexes"], "both applied, in order");
    let second = apply_migrations(port, migs).await;
    assert!(second.is_empty(), "the ledger made pass two a no-op (idempotent)");
});
println!("Exercise 8.1 passed");


## 9. sqlx in the Ecosystem

Runtime SQL vs the macros is a question of **when** things are checked: the macros (`query!`, `query_as!`) read your SQL at compile time against a live `DATABASE_URL`; the runtime API checks at run time, with the same types, pool, and transactions. The call sites differ by one `!`.

**See**: §9 of the example notebook.


In [ ]:
// 9.1 The runtime form of the query the macro form would compile-check.
// let rows = sqlx::query_as!(Fill, "SELECT id, name FROM t")
//     .fetch_all(&pool)
//     .await
//     .unwrap();
// — the macro form: the same chain plus `!`, checked against DATABASE_URL
//   while `cargo build` ran (the `macros` feature is off in this notebook).

/// The RUNTIME form: decode `SELECT * FROM t` into `Fill`s with query_as
/// and return the row count. Switching to the macro form above would change
/// the checking time, not a single type or method.
async fn macro_equivalent(port: u16) -> usize {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    insert_named(port, "ana").await;
    insert_named(port, "bob").await;
    assert_eq!(macro_equivalent(port).await, 2, "same chain, same types — only the check moved");
});
println!("Exercise 9.1 passed");


## 10. Mixed Review

Four problems, each from a different corner of the chapter.

**See**: §2 and §3 of the example notebook (10.1), §4 and §5 (10.2), §6 (10.3), §7 and §2's write receipts (10.4).


In [ ]:
// 10.1 Filtered query_as: the lookup and the decode in one chain.

/// Decode `SELECT * FROM t WHERE name = $1` into `Fill`s. An empty result
/// is an empty vec — fetch_all never errors on zero rows.
async fn fills_by_name(port: u16, name: &str) -> Vec<Fill> {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    insert_named(port, "ana").await;
    insert_named(port, "ana").await;
    insert_named(port, "bob").await;
    let fills = fills_by_name(port, "ana").await;
    assert_eq!(fills.len(), 2);
    assert!(fills.iter().all(|f| f.name.as_deref() == Some("ana")));
    assert_eq!(fills.iter().map(|f| f.id).collect::<Vec<i32>>(), vec![1, 2],
        "the desk returns rows id-ordered");
    assert!(fills_by_name(port, "nobody").await.is_empty(), "empty result, not an error");
});
println!("Exercise 10.1 passed");


In [ ]:
// 10.2 A NULL insert whose survival depends on the transaction's exit.

/// Begin a transaction, INSERT one NULL name inside it, then COMMIT if
/// `commit_it` is true else ROLLBACK. Return every name in the table
/// afterward (Option<String> per row).
async fn tx_null(port: u16, commit_it: bool) -> Vec<Option<String>> {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let kept = tx_null(port, true).await;
    assert_eq!(kept.iter().filter(|n| n.is_none()).count(), 1, "the committed NULL is visible");
    let gone = tx_null(port, false).await;
    assert_eq!(gone, kept, "the rolled-back NULL left no row behind");
});
println!("Exercise 10.2 passed");


In [ ]:
// 10.3 A batch of hostile payloads: every one a bound parameter.

/// For each payload in `payloads`: INSERT it as a bound parameter, then
/// look it up by the same bound parameter. Return how many payloads
/// matched themselves (all of them — values never touch the SQL text).
async fn hostile_batch(port: u16, payloads: &[&str]) -> usize {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let payloads: &[&str] = &["ok-1", "", "x'; DROP TABLE t; --", "1000'); --"];
    assert_eq!(hostile_batch(port, payloads).await, 4, "every payload matched itself");
    assert_eq!(table_alive(port).await, 4, "the table survived the whole batch");
});
println!("Exercise 10.3 passed");


In [ ]:
// 10.4 Write under pressure: the pool says no, then yes.

/// On a 1-connection pool (acquire_timeout 200ms): hold the only
/// connection, attempt `INSERT INTO t (name) VALUES ('under-pressure')`
/// (it must fail with PoolTimedOut), release the connection, retry. Return
/// rows_affected of the insert that landed.
async fn write_under_pressure(port: u16) -> u64 {
    todo!()
}

// --- check (do not modify) ---
let port = pg_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    assert_eq!(write_under_pressure(port).await, 1,
        "the retry landed exactly one row once the socket freed up");
});
println!("Exercise 10.4 passed");


## 11. What Comes Next

| Next | What it borrows from here |
|---|---|
| **20 — PyO3** | the boundary crosses languages: `Fill`-style row structs become `#[pyclass]`es; decode-at-the-boundary and the error layering are the export contract. |
| **21 — Capstone Prep** | §7's pool knobs and the §2 error shapes become the capstone's load and failure tests; §8's migration ledger is its startup sequence. |
| **22 — Capstone** | §5's transactions persist the fill feed behind chapter 18's cache; bound parameters and pool timeouts are the production posture. |

Carry forward: **the pool is configured once and cloned everywhere; values are always bound, never formatted into SQL; the transaction is a value whose drop is a rollback; NULL is Option, and the error variant tells you which layer failed.**
